# 04 — Validazione della diarizzazione

Questo notebook viene eseguito **dopo** la diarizzazione batch.

Obiettivi:

1. verificare che tutti i risultati siano presenti;
2. costruire un riepilogo per registrazione e canale;
3. individuare automaticamente i casi da controllare;
4. preparare una coda di ascolto con i segmenti più rappresentativi;
5. creare una tabella per l'assegnazione manuale dei ruoli degli speaker.


In [1]:
from pathlib import Path
import re
import pandas as pd
import numpy as np
import soundfile as sf
from IPython.display import Audio, display

# Cartelle del progetto
PROJECT_DIR = Path.cwd()
AUDIO_DIR = PROJECT_DIR / "file_wav"
DIAR_DIR = PROJECT_DIR / "risultati" / "diarizzazione_batch" / "completo"
STATE_CSV = DIAR_DIR / "stato_elaborazione.csv"

OUTPUT_DIR = PROJECT_DIR / "risultati" / "validazione_diarizzazione"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("AUDIO_DIR:", AUDIO_DIR)
print("DIAR_DIR:", DIAR_DIR)
print("STATE_CSV esiste:", STATE_CSV.exists())


PROJECT_DIR: c:\Users\acer\Desktop\ProgettoTesi
AUDIO_DIR: c:\Users\acer\Desktop\ProgettoTesi\file_wav
DIAR_DIR: c:\Users\acer\Desktop\ProgettoTesi\risultati\diarizzazione_batch\completo
STATE_CSV esiste: True


## 1. Controllo del registro batch

La cella seguente legge il registro finale e mostra le colonne disponibili.  
Serve anche a verificare che risultino **185 elaborazioni completate**.


In [2]:
if not STATE_CSV.exists():
    raise FileNotFoundError(
        f"Registro non trovato: {STATE_CSV}\n"
        "Controlla di avviare il notebook dalla cartella principale del progetto."
    )

df_stato = pd.read_csv(STATE_CSV)

print("Numero righe:", len(df_stato))
print("Colonne:", list(df_stato.columns))
display(df_stato.head())

if "stato" in df_stato.columns:
    print("\nConteggio degli stati:")
    display(df_stato["stato"].value_counts(dropna=False).rename_axis("stato").to_frame("conteggio"))


Numero righe: 185
Colonne: ['nome_file', 'canale', 'stato', 'numero_segmenti', 'numero_speaker', 'tempo_secondi', 'errore']


,nome_file,canale,stato,numero_segmenti,numero_speaker,tempo_secondi,errore
0,ID1.StudioRuggi-006.wav,sinistro,completato,73,2,312.85,NaN
1,ID1.StudioRuggi-006.wav,destro,completato,73,2,326.04,NaN
2,ID10.StudioRuggi.wav,sinistro,completato,64,2,199.51,NaN
3,ID10.StudioRuggi.wav,destro,completato,62,2,197.17,NaN
4,ID101.StudioRuggi.wav,sinistro,completato,96,2,278.28,NaN



Conteggio degli stati:


,conteggio
stato,
completato,185


## 2. Ricerca automatica dei file contenenti i segmenti

Il codice cerca ricorsivamente:

- file CSV dei segmenti;
- in alternativa, file RTTM standard prodotti dalla diarizzazione.

Il notebook prova a riconoscere automaticamente nomi di colonne come
`inizio_secondi`, `fine_secondi`, `durata_secondi`, `speaker` e `canale`.


In [3]:
def trova_colonna(colonne, candidati):
    lookup = {str(c).strip().lower(): c for c in colonne}
    for candidato in candidati:
        if candidato.lower() in lookup:
            return lookup[candidato.lower()]
    return None


def inferisci_canale_da_nome(nome):
    nome_l = nome.lower()
    if "sinistro" in nome_l or "_left" in nome_l or "-left" in nome_l:
        return "sinistro"
    if "destro" in nome_l or "_right" in nome_l or "-right" in nome_l:
        return "destro"
    if "mono" in nome_l:
        return "mono"
    return "non_specificato"


def inferisci_recording_id(nome):
    match = re.search(r"(ID\d+)", nome, flags=re.IGNORECASE)
    return match.group(1).upper() if match else Path(nome).stem


def leggi_csv_segmenti(percorso):
    df = pd.read_csv(percorso)

    col_speaker = trova_colonna(
        df.columns,
        ["speaker", "speaker_id", "label_speaker", "parlante"]
    )
    col_start = trova_colonna(
        df.columns,
        ["inizio_secondi", "start", "start_s", "inizio", "segment_start"]
    )
    col_end = trova_colonna(
        df.columns,
        ["fine_secondi", "end", "end_s", "fine", "segment_end"]
    )
    col_duration = trova_colonna(
        df.columns,
        ["durata_secondi", "duration", "duration_s", "durata", "segment_duration"]
    )

    # Un CSV viene considerato tabella di segmenti solo se contiene speaker
    # e almeno due informazioni temporali sufficienti.
    if col_speaker is None:
        return None

    if col_start is None and col_end is None and col_duration is None:
        return None

    out = pd.DataFrame()
    out["speaker"] = df[col_speaker].astype(str)

    if col_start is not None:
        out["inizio_secondi"] = pd.to_numeric(df[col_start], errors="coerce")
    else:
        out["inizio_secondi"] = np.nan

    if col_end is not None:
        out["fine_secondi"] = pd.to_numeric(df[col_end], errors="coerce")
    else:
        out["fine_secondi"] = np.nan

    if col_duration is not None:
        out["durata_secondi"] = pd.to_numeric(df[col_duration], errors="coerce")
    else:
        out["durata_secondi"] = out["fine_secondi"] - out["inizio_secondi"]

    # Ricostruzione dell'estremo mancante, quando possibile
    mask_end = out["fine_secondi"].isna()
    out.loc[mask_end, "fine_secondi"] = (
        out.loc[mask_end, "inizio_secondi"]
        + out.loc[mask_end, "durata_secondi"]
    )

    mask_start = out["inizio_secondi"].isna()
    out.loc[mask_start, "inizio_secondi"] = (
        out.loc[mask_start, "fine_secondi"]
        - out.loc[mask_start, "durata_secondi"]
    )

    col_recording = trova_colonna(
        df.columns,
        ["recording_id", "audio_id", "nome_file", "file", "filename", "audio_file"]
    )
    col_channel = trova_colonna(
        df.columns,
        ["canale", "channel", "audio_channel"]
    )

    if col_recording is not None:
        out["recording_id"] = df[col_recording].astype(str)
    else:
        out["recording_id"] = inferisci_recording_id(percorso.stem)

    if col_channel is not None:
        out["canale"] = df[col_channel].astype(str).str.lower()
    else:
        out["canale"] = inferisci_canale_da_nome(percorso.stem)

    out["file_risultato"] = percorso.name
    out["percorso_risultato"] = str(percorso)

    out = out.dropna(
        subset=["speaker", "inizio_secondi", "fine_secondi", "durata_secondi"]
    ).copy()

    out = out[
        (out["durata_secondi"] > 0)
        & (out["fine_secondi"] > out["inizio_secondi"])
    ].copy()

    return out


csv_files = [
    p for p in DIAR_DIR.rglob("*.csv")
    if p.name.lower() != "stato_elaborazione.csv"
]

print(f"CSV candidati trovati: {len(csv_files)}")

tabelle = []
csv_ignorati = []

for percorso in csv_files:
    try:
        tabella = leggi_csv_segmenti(percorso)
        if tabella is not None and not tabella.empty:
            tabelle.append(tabella)
        else:
            csv_ignorati.append(percorso.name)
    except Exception as exc:
        print(f"[AVVISO] Impossibile leggere {percorso.name}: {exc}")
        csv_ignorati.append(percorso.name)

if tabelle:
    df_segmenti = pd.concat(tabelle, ignore_index=True)
else:
    df_segmenti = pd.DataFrame()

print("Tabelle di segmenti riconosciute:", len(tabelle))
print("Segmenti caricati:", len(df_segmenti))
print("CSV ignorati:", len(csv_ignorati))

if not df_segmenti.empty:
    display(df_segmenti.head())


CSV candidati trovati: 185
Tabelle di segmenti riconosciute: 185
Segmenti caricati: 19722
CSV ignorati: 0


,speaker,inizio_secondi,fine_secondi,durata_secondi,recording_id,canale,file_risultato,percorso_risultato
0,SPEAKER_00,0.368,1.027,0.658,ID1.StudioRuggi-006,destro,ID1.StudioRuggi-006__destro.csv,c:\Users\acer\Desktop\ProgettoTesi\risultati\d...
1,SPEAKER_01,1.567,3.338,1.772,ID1.StudioRuggi-006,destro,ID1.StudioRuggi-006__destro.csv,c:\Users\acer\Desktop\ProgettoTesi\risultati\d...
2,SPEAKER_00,4.064,5.380,1.316,ID1.StudioRuggi-006,destro,ID1.StudioRuggi-006__destro.csv,c:\Users\acer\Desktop\ProgettoTesi\risultati\d...
3,SPEAKER_01,5.903,7.878,1.974,ID1.StudioRuggi-006,destro,ID1.StudioRuggi-006__destro.csv,c:\Users\acer\Desktop\ProgettoTesi\risultati\d...
4,SPEAKER_00,6.713,7.439,0.726,ID1.StudioRuggi-006,destro,ID1.StudioRuggi-006__destro.csv,c:\Users\acer\Desktop\ProgettoTesi\risultati\d...


### Lettura RTTM di riserva

Questa cella viene usata soltanto se nella cartella non sono stati trovati CSV dei segmenti.


In [4]:
def leggi_rttm(percorso):
    righe = []

    with open(percorso, "r", encoding="utf-8") as file:
        for linea in file:
            parti = linea.strip().split()

            if len(parti) < 8 or parti[0] != "SPEAKER":
                continue

            start = float(parti[3])
            durata = float(parti[4])

            righe.append({
                "recording_id": inferisci_recording_id(percorso.stem),
                "canale": inferisci_canale_da_nome(percorso.stem),
                "inizio_secondi": start,
                "fine_secondi": start + durata,
                "durata_secondi": durata,
                "speaker": parti[7],
                "file_risultato": percorso.name,
                "percorso_risultato": str(percorso),
            })

    return pd.DataFrame(righe)


if df_segmenti.empty:
    rttm_files = list(DIAR_DIR.rglob("*.rttm"))
    print("RTTM trovati:", len(rttm_files))

    tabelle_rttm = []
    for percorso in rttm_files:
        tabella = leggi_rttm(percorso)
        if not tabella.empty:
            tabelle_rttm.append(tabella)

    if tabelle_rttm:
        df_segmenti = pd.concat(tabelle_rttm, ignore_index=True)
        print("Segmenti caricati dagli RTTM:", len(df_segmenti))
        display(df_segmenti.head())
    else:
        print(
            "Nessuna tabella di segmenti riconosciuta.\n"
            "Esegui la cella diagnostica finale del notebook e condividi l'output."
        )


## 3. Riepilogo per registrazione e canale

Vengono calcolati:

- numero di segmenti;
- numero di speaker;
- durata totale del parlato rilevato;
- durata media e massima dei segmenti;
- flag per i casi con numero di speaker diverso da 2.


In [5]:
if df_segmenti.empty:
    raise RuntimeError(
        "Non sono stati trovati segmenti utilizzabili. "
        "Controlla la struttura dei file nella cartella di output."
    )

df_segmenti["recording_id"] = df_segmenti["recording_id"].astype(str)
df_segmenti["canale"] = df_segmenti["canale"].astype(str).str.lower()
df_segmenti["speaker"] = df_segmenti["speaker"].astype(str)

riepilogo = (
    df_segmenti
    .groupby(["recording_id", "canale"], as_index=False)
    .agg(
        numero_segmenti=("speaker", "size"),
        numero_speaker=("speaker", "nunique"),
        parlato_totale_secondi=("durata_secondi", "sum"),
        durata_media_segmento=("durata_secondi", "mean"),
        durata_massima_segmento=("durata_secondi", "max"),
    )
)

riepilogo["parlato_totale_secondi"] = riepilogo["parlato_totale_secondi"].round(2)
riepilogo["durata_media_segmento"] = riepilogo["durata_media_segmento"].round(3)
riepilogo["durata_massima_segmento"] = riepilogo["durata_massima_segmento"].round(3)

riepilogo["speaker_anomali"] = riepilogo["numero_speaker"] != 2

# Differenza nel numero di speaker tra i due canali della stessa registrazione
diff_canali = (
    riepilogo
    .groupby("recording_id")["numero_speaker"]
    .agg(["min", "max"])
    .reset_index()
)
diff_canali["discordanza_canali"] = diff_canali["min"] != diff_canali["max"]

riepilogo = riepilogo.merge(
    diff_canali[["recording_id", "discordanza_canali"]],
    on="recording_id",
    how="left"
)

riepilogo["da_revisionare"] = (
    riepilogo["speaker_anomali"]
    | riepilogo["discordanza_canali"]
)

percorso_riepilogo = OUTPUT_DIR / "riepilogo_diarizzazione.csv"
riepilogo.to_csv(percorso_riepilogo, index=False)

print("Righe del riepilogo:", len(riepilogo))
print("Casi canale da revisionare:", int(riepilogo["da_revisionare"].sum()))
print("File salvato in:", percorso_riepilogo)

display(
    riepilogo.sort_values(
        ["da_revisionare", "numero_speaker", "recording_id"],
        ascending=[False, False, True]
    )
)


Righe del riepilogo: 185
Casi canale da revisionare: 30
File salvato in: c:\Users\acer\Desktop\ProgettoTesi\risultati\validazione_diarizzazione\riepilogo_diarizzazione.csv


,recording_id,canale,numero_segmenti,numero_speaker,parlato_totale_secondi,durata_media_segmento,durata_massima_segmento,speaker_anomali,discordanza_canali,da_revisionare
89,ID30.StudioRuggi,destro,171,4,358.18,2.095,18.698,True,False,True
90,ID30.StudioRuggi,sinistro,164,4,351.50,2.143,20.739,True,False,True
13,ID112.StudioRuggi,destro,117,3,271.00,2.316,11.526,True,False,True
14,ID112.StudioRuggi,sinistro,116,3,276.51,2.384,11.492,True,False,True
21,ID118.StudioRuggi,destro,124,3,201.03,1.621,17.246,True,False,True
...,...,...,...,...,...,...,...,...,...,...
178,ID90.StudioRuggi,sinistro,69,2,121.99,1.768,9.298,False,False,False
179,ID91.StudioRuggi,destro,89,2,121.69,1.367,6.733,False,False,False
180,ID91.StudioRuggi,sinistro,96,2,137.01,1.427,6.699,False,False,False
181,ID92.StudioRuggi,destro,116,2,226.60,1.953,13.146,False,False,False


## 4. Elenco dei casi prioritari

La tabella contiene le registrazioni in cui:

- il numero di speaker è diverso da 2;
- i canali sinistro e destro non concordano sul numero di speaker.


In [6]:
casi_revisionare = (
    riepilogo[riepilogo["da_revisionare"]]
    .sort_values(
        ["numero_speaker", "recording_id", "canale"],
        ascending=[False, True, True]
    )
    .reset_index(drop=True)
)

percorso_casi = OUTPUT_DIR / "casi_da_revisionare.csv"
casi_revisionare.to_csv(percorso_casi, index=False)

print("Registrazioni uniche da revisionare:",
      casi_revisionare["recording_id"].nunique())
print("Righe canale da revisionare:", len(casi_revisionare))
print("File salvato in:", percorso_casi)

display(casi_revisionare)


Registrazioni uniche da revisionare: 15
Righe canale da revisionare: 30
File salvato in: c:\Users\acer\Desktop\ProgettoTesi\risultati\validazione_diarizzazione\casi_da_revisionare.csv


,recording_id,canale,numero_segmenti,numero_speaker,parlato_totale_secondi,durata_media_segmento,durata_massima_segmento,speaker_anomali,discordanza_canali,da_revisionare
0,ID30.StudioRuggi,destro,171,4,358.18,2.095,18.698,True,False,True
1,ID30.StudioRuggi,sinistro,164,4,351.50,2.143,20.739,True,False,True
2,ID112.StudioRuggi,destro,117,3,271.00,2.316,11.526,True,False,True
3,ID112.StudioRuggi,sinistro,116,3,276.51,2.384,11.492,True,False,True
4,ID118.StudioRuggi,destro,124,3,201.03,1.621,17.246,True,False,True
5,ID118.StudioRuggi,sinistro,111,3,199.08,1.794,17.246,True,False,True
6,ID121.StudioRuggi,destro,114,3,195.19,1.712,8.488,True,False,True
7,ID121.StudioRuggi,sinistro,121,3,198.07,1.637,8.573,True,False,True
8,ID155.StudioRuggi,destro,142,3,272.41,1.918,8.657,True,False,True
9,ID155.StudioRuggi,sinistro,143,3,271.35,1.898,8.657,True,False,True


## 5. Segmenti rappresentativi per ciascuno speaker

Per ogni combinazione registrazione–canale–speaker vengono selezionati i tre segmenti più lunghi.  
Questi segmenti sono normalmente più facili da ascoltare e usare per riconoscere il ruolo.


In [7]:
campioni = (
    df_segmenti
    .sort_values("durata_secondi", ascending=False)
    .groupby(["recording_id", "canale", "speaker"], as_index=False)
    .head(3)
    .sort_values(
        ["recording_id", "canale", "speaker", "durata_secondi"],
        ascending=[True, True, True, False]
    )
    .reset_index(drop=True)
)

campioni["ruolo_manual"] = ""
campioni["note"] = ""

percorso_campioni = OUTPUT_DIR / "campioni_rappresentativi.csv"
campioni.to_csv(percorso_campioni, index=False)

print("Campioni selezionati:", len(campioni))
print("File salvato in:", percorso_campioni)
display(campioni.head(20))


Campioni selezionati: 1180
File salvato in: c:\Users\acer\Desktop\ProgettoTesi\risultati\validazione_diarizzazione\campioni_rappresentativi.csv


,speaker,inizio_secondi,fine_secondi,durata_secondi,recording_id,canale,file_risultato,percorso_risultato,ruolo_manual,note
0,SPEAKER_00,244.010,249.933,5.923,ID1.StudioRuggi-006,destro,ID1.StudioRuggi-006__destro.csv,c:\Users\acer\Desktop\ProgettoTesi\risultati\d...,,
1,SPEAKER_00,305.992,311.003,5.012,ID1.StudioRuggi-006,destro,ID1.StudioRuggi-006__destro.csv,c:\Users\acer\Desktop\ProgettoTesi\risultati\d...,,
2,SPEAKER_00,271.972,276.460,4.489,ID1.StudioRuggi-006,destro,ID1.StudioRuggi-006__destro.csv,c:\Users\acer\Desktop\ProgettoTesi\risultati\d...,,
3,SPEAKER_01,127.066,153.323,26.258,ID1.StudioRuggi-006,destro,ID1.StudioRuggi-006__destro.csv,c:\Users\acer\Desktop\ProgettoTesi\risultati\d...,,
4,SPEAKER_01,223.658,241.883,18.225,ID1.StudioRuggi-006,destro,ID1.StudioRuggi-006__destro.csv,c:\Users\acer\Desktop\ProgettoTesi\risultati\d...,,
5,SPEAKER_01,169.675,185.842,16.166,ID1.StudioRuggi-006,destro,ID1.StudioRuggi-006__destro.csv,c:\Users\acer\Desktop\ProgettoTesi\risultati\d...,,
6,SPEAKER_00,243.993,249.916,5.923,ID1.StudioRuggi-006,sinistro,ID1.StudioRuggi-006__sinistro.csv,c:\Users\acer\Desktop\ProgettoTesi\risultati\d...,,
7,SPEAKER_00,305.941,311.105,5.164,ID1.StudioRuggi-006,sinistro,ID1.StudioRuggi-006__sinistro.csv,c:\Users\acer\Desktop\ProgettoTesi\risultati\d...,,
8,SPEAKER_00,271.972,276.443,4.472,ID1.StudioRuggi-006,sinistro,ID1.StudioRuggi-006__sinistro.csv,c:\Users\acer\Desktop\ProgettoTesi\risultati\d...,,
9,SPEAKER_01,158.167,185.167,27.000,ID1.StudioRuggi-006,sinistro,ID1.StudioRuggi-006__sinistro.csv,c:\Users\acer\Desktop\ProgettoTesi\risultati\d...,,


## 6. Funzioni di ascolto

La prima funzione cerca automaticamente il WAV associato a un `recording_id`.  
Per gli ID con più registrazioni, è possibile indicare direttamente il nome del file.


In [8]:
def trova_audio(recording_id=None, nome_file=None):
    if nome_file is not None:
        percorso = AUDIO_DIR / nome_file
        if not percorso.exists():
            raise FileNotFoundError(percorso)
        return percorso

    candidati = sorted(AUDIO_DIR.glob(f"{recording_id}*.wav"))

    if not candidati:
        # Ricerca più permissiva
        candidati = sorted(
            p for p in AUDIO_DIR.glob("*.wav")
            if str(recording_id).lower() in p.stem.lower()
        )

    if not candidati:
        raise FileNotFoundError(
            f"Nessun WAV trovato per {recording_id}."
        )

    if len(candidati) > 1:
        print("Sono presenti più registrazioni:")
        for indice, percorso in enumerate(candidati):
            print(f"  [{indice}] {percorso.name}")
        print("Passa nome_file='...' per scegliere quella corretta.")

    return candidati[0]


def indice_canale(canale, numero_canali):
    canale = str(canale).lower()

    if numero_canali == 1 or canale == "mono":
        return 0
    if canale == "sinistro":
        return 0
    if canale == "destro":
        return 1

    match = re.search(r"(\d+)", canale)
    if match:
        indice = int(match.group(1)) - 1
        if 0 <= indice < numero_canali:
            return indice

    raise ValueError(f"Canale non riconosciuto: {canale}")


def ascolta_intervallo(
    recording_id,
    canale,
    inizio_secondi,
    fine_secondi,
    nome_file=None,
    margine_secondi=0.15
):
    percorso = trova_audio(
        recording_id=recording_id,
        nome_file=nome_file
    )

    audio, sample_rate = sf.read(
        percorso,
        always_2d=True,
        dtype="float32"
    )

    idx = indice_canale(canale, audio.shape[1])

    inizio = max(0.0, float(inizio_secondi) - margine_secondi)
    fine = min(
        len(audio) / sample_rate,
        float(fine_secondi) + margine_secondi
    )

    campione_inizio = int(inizio * sample_rate)
    campione_fine = int(fine * sample_rate)

    segmento = audio[campione_inizio:campione_fine, idx]

    print(
        f"{percorso.name} | {canale} | "
        f"{inizio:.2f}-{fine:.2f} s"
    )
    display(Audio(segmento, rate=sample_rate))


def ascolta_speaker(
    recording_id,
    canale,
    speaker,
    numero_campioni=3,
    nome_file=None
):
    selezione = campioni[
        (campioni["recording_id"].astype(str) == str(recording_id))
        & (campioni["canale"].astype(str).str.lower() == str(canale).lower())
        & (campioni["speaker"].astype(str) == str(speaker))
    ].head(numero_campioni)

    if selezione.empty:
        print("Nessun segmento trovato.")
        return

    for _, riga in selezione.iterrows():
        print(
            f"\nSpeaker: {riga['speaker']} | "
            f"durata: {riga['durata_secondi']:.2f} s"
        )
        ascolta_intervallo(
            recording_id=recording_id,
            canale=canale,
            inizio_secondi=riga["inizio_secondi"],
            fine_secondi=riga["fine_secondi"],
            nome_file=nome_file
        )


## 7. Tabella per l'assegnazione dei ruoli

Per ciascuno speaker inserire nella colonna `ruolo` uno dei valori:

- `paziente`
- `medico`
- `voce_registrata`
- `terza_persona`
- `rumore`
- `incerto`


In [9]:
mappa_ruoli = (
    df_segmenti[
        ["recording_id", "canale", "speaker"]
    ]
    .drop_duplicates()
    .sort_values(["recording_id", "canale", "speaker"])
    .reset_index(drop=True)
)

mappa_ruoli["ruolo"] = ""
mappa_ruoli["confidenza"] = ""
mappa_ruoli["note"] = ""

percorso_mappa = OUTPUT_DIR / "mappa_ruoli_speaker.csv"

# Non sovrascrive una tabella già compilata.
if not percorso_mappa.exists():
    mappa_ruoli.to_csv(percorso_mappa, index=False)
    print("Template creato in:", percorso_mappa)
else:
    print(
        "Il template esiste già e non è stato sovrascritto:",
        percorso_mappa
    )

display(mappa_ruoli.head(20))


Template creato in: c:\Users\acer\Desktop\ProgettoTesi\risultati\validazione_diarizzazione\mappa_ruoli_speaker.csv


,recording_id,canale,speaker,ruolo,confidenza,note
0,ID1.StudioRuggi-006,destro,SPEAKER_00,,,
1,ID1.StudioRuggi-006,destro,SPEAKER_01,,,
2,ID1.StudioRuggi-006,sinistro,SPEAKER_00,,,
3,ID1.StudioRuggi-006,sinistro,SPEAKER_01,,,
4,ID10.StudioRuggi,destro,SPEAKER_00,,,
5,ID10.StudioRuggi,destro,SPEAKER_01,,,
6,ID10.StudioRuggi,sinistro,SPEAKER_00,,,
7,ID10.StudioRuggi,sinistro,SPEAKER_01,,,
8,ID101.StudioRuggi,destro,SPEAKER_00,,,
9,ID101.StudioRuggi,destro,SPEAKER_01,,,


## 8. Diagnostica della struttura dei file

Eseguire questa cella soltanto se il notebook non riesce a riconoscere i file dei segmenti.  
L'output permette di adattare rapidamente il caricamento alla struttura reale.


In [10]:
print("Contenuto della cartella di diarizzazione:\n")

for percorso in sorted(DIAR_DIR.rglob("*")):
    if percorso.is_file():
        print(percorso.relative_to(DIAR_DIR))

print("\nPrime colonne dei CSV:\n")

for percorso in sorted(DIAR_DIR.rglob("*.csv"))[:20]:
    try:
        prova = pd.read_csv(percorso, nrows=3)
        print(percorso.name, "->", list(prova.columns))
    except Exception as exc:
        print(percorso.name, "-> ERRORE:", exc)


Contenuto della cartella di diarizzazione:

ID1.StudioRuggi-006__destro.csv
ID1.StudioRuggi-006__sinistro.csv
ID10.StudioRuggi__destro.csv
ID10.StudioRuggi__sinistro.csv
ID101.StudioRuggi__destro.csv
ID101.StudioRuggi__sinistro.csv
ID103.StudioRuggi__mono.csv
ID104.StudioRuggi__destro.csv
ID104.StudioRuggi__sinistro.csv
ID108.StudioRuggi__destro.csv
ID108.StudioRuggi__sinistro.csv
ID11.StudioRuggi__destro.csv
ID11.StudioRuggi__sinistro.csv
ID112.StudioRuggi__destro.csv
ID112.StudioRuggi__sinistro.csv
ID114.StudioRuggi__destro.csv
ID114.StudioRuggi__sinistro.csv
ID116.StudioRuggi__destro.csv
ID116.StudioRuggi__sinistro.csv
ID117.StudioRuggi__destro.csv
ID117.StudioRuggi__sinistro.csv
ID118.StudioRuggi__destro.csv
ID118.StudioRuggi__sinistro.csv
ID12.StudioRuggi__destro.csv
ID12.StudioRuggi__sinistro.csv
ID121.StudioRuggi__destro.csv
ID121.StudioRuggi__sinistro.csv
ID122.StudioRuggi__destro.csv
ID122.StudioRuggi__sinistro.csv
ID126.StudioRuggi__destro.csv
ID126.StudioRuggi__sinistro.csv
